# Smart MCQ Solver - DL & GenAI Project

# 1. Environment Setup

Import all the required libraries that will be used throughout this project.

In [ ]:
import os
import re
import json
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

print("Environment Ready ✅")

# 2. Configuration

Define project constants and file paths used throughout the notebook.

In [ ]:
SEED = 42
VAL_SIZE = 0.2

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_SUBMISSION_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

OPTIONS = ["A", "B", "C", "D", "E"]

# 3. Load the Dataset

Load the training and test datasets and inspect their basic structure before performing any preprocessing or modeling.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

train.head()

# 4. Dataset Overview

Understand the structure, data types, and completeness of the dataset.

In [ ]:
train.info()

# 5. NLP 

Machine learning models cannot understand raw text. Before applying any learning algorithm, textual data must be converted into numerical representations. In this milestone, we build progressively better text representations, starting from TF-IDF and Word2Vec.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

text_columns = ["prompt", "A", "B", "C", "D", "E"]

for col in text_columns:
    train[col] = train[col].apply(clean_text)
    test[col] = test[col].apply(clean_text)

## Tokenization

Tokenization is the process of breaking raw text into smaller units called tokens. These tokens form the basic input for almost every NLP model, from TF-IDF to modern Transformer architectures.

In [ ]:
sample_text = train.loc[0, "prompt"]

tokens = sample_text.split()

print("Original Text:\n")
print(sample_text)

print("\nTokens:\n")
print(tokens)

## TF-IDF (Term Frequency - Inverse Document Frequency)

Bag of Words treats every word as equally important. However, common words such as "is", "the", and "of" appear in almost every document and contribute very little information.

TF-IDF improves upon Bag of Words by assigning higher weights to important words while reducing the influence of very common words.

### TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_corpus = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

tfidf_matrix = tfidf.fit_transform(train_corpus)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

### Similarity using TF-IDF

Once every document has been converted into TF-IDF vectors, we can compare two pieces of text by measuring how similar their vectors are. Cosine Similarity is the most commonly used metric for this purpose.

In [ ]:
sample_prompt = train.loc[0, "prompt"]

option_vectors = tfidf.transform(train.loc[[0], ["A", "B", "C", "D", "E"]].values.flatten())
prompt_vector = tfidf.transform([sample_prompt])

scores = cosine_similarity(prompt_vector, option_vectors).flatten()

for option, score in zip(OPTIONS, scores):
    print(f"{option}: {score:.4f}")

## Word2Vec Embeddings

Unlike TF-IDF, which represents words using frequency statistics, Word2Vec learns dense vector representations where semantically similar words are placed closer together in the embedding space.

For this project, Word2Vec serves as a conceptual improvement over TF-IDF before moving to Transformer-based embeddings.

In [ ]:
from gensim.models import Word2Vec

sentences = [text.split() for text in train["prompt"]]

word2vec = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    seed=SEED
)

print(word2vec.wv.most_similar("following", topn=5))

## Evaluation Metric - Mean Average Precision @ 3 (MAP@3)

The competition is evaluated using MAP@3 instead of accuracy. Since each prediction consists of the top three ranked answer choices, this metric rewards models that rank the correct answer higher.

In [ ]:
def average_precision_at_3(actual, predicted):
    if actual in predicted[:3]:
        return 1 / (predicted[:3].index(actual) + 1)
    return 0

def map_at_3(actuals, predictions):
    scores = [
        average_precision_at_3(a, p)
        for a, p in zip(actuals, predictions)
    ]
    return np.mean(scores)